# Batch human-video retargeting

Objective: reuse the successful release-v3 controller for multiple human transfers in a fixed simulator scene. Run in the working notebook-05 Colab environment. Tracking and simulation are automatic; event annotation and replay approval remain explicit. Run sections deliberately: do not approve unseen videos by running the notebook top to bottom.

In [ ]:
!git clone https://github.com/huggingface/lerobot.git
%cd lerobot
!pip install -e ".[libero]"

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys, json
project = Path('/content/drive/MyDrive/Projects/humanoid-challenge')
runner = project / 'scripts/batch_retarget.py'
def batch(*args):
    subprocess.run([sys.executable, str(runner), '--project', str(project), *map(str,args)], check=True)
assert runner.exists()


## Register and track
Use filenames demo_001.mov, demo_002.mp4, etc. Verify calibration settings before registering. Tracking logs are saved under outputs/batch_logs.

In [ ]:
videos = sorted(p for p in (project/'data/raw').glob('demo_*') if p.suffix.lower() in {'.mov','.mp4','.m4v'})
assert videos, 'Add demo_* videos to data/raw first.'
calibration = 'outputs/calibration/iphone13mini_1080x1920_provisional.npz'
print([p.name for p in videos])
batch('register', '--calibration', calibration, '--videos', *[str(p.relative_to(project)) for p in videos])
batch('track')
batch('status')


## Annotate one video
Select a demo. This helper shows exact decoded frames, and saves only events you explicitly choose. If an event is ambiguous, inspect nearby frames. New videos need their own six events. Rerun this section for each demo.

In [ ]:
demo = 'demo_016'
manifest = json.loads((project/'configs/retarget_batch.json').read_text())
item = manifest['demos'][demo]
folder = project/'data/processed'/demo
events_path = folder/'human_events.json'
annotations = json.loads(events_path.read_text())
tracking = json.loads((folder/'tracking_report.json').read_text())

from IPython.display import display
import ipywidgets as widgets
import cv2
import hashlib, tempfile, shutil, math

# Close the previous viewer before replacing its callbacks/state.
for old_widget in globals().get('_annotation_widgets', []):
    old_widget.close()

# Copy from Drive once, then decode ONCE. Keep compact JPEG previews in memory.
source = project/item['video']
stat = source.stat()
cache_key = (str(source), stat.st_size, stat.st_mtime_ns,
             item.get('tracking_inputs', {}).get('video'))
if globals().get('_annotation_cache', {}).get('key') != cache_key:
    cache_dir = Path(tempfile.gettempdir())/'humanoid_annotation'
    cache_dir.mkdir(exist_ok=True)
    local_video = cache_dir/('current'+source.suffix)
    print('Copying video locally and preparing previews once…', flush=True)
    shutil.copyfile(source, local_video)
    with local_video.open('rb') as f:
        video_hash = hashlib.file_digest(f, 'sha256').hexdigest()
    assert video_hash == item['tracking_inputs']['video'], 'Video changed: rerun tracking before annotation.'
    cap = cv2.VideoCapture(str(local_video))
    cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)
    previews, timestamps = [], []
    try:
        while True:
            ok, frame = cap.read()
            if not ok: break
            timestamps.append(cap.get(cv2.CAP_PROP_POS_MSEC)/1000)
            height, width = frame.shape[:2]
            scale = min(480/width, 640/height, 1.0)
            preview = cv2.resize(frame, (round(width*scale), round(height*scale)), interpolation=cv2.INTER_AREA)
            ok, encoded = cv2.imencode('.jpg', preview, [cv2.IMWRITE_JPEG_QUALITY, 85])
            assert ok
            previews.append(encoded.tobytes())
    finally:
        cap.release()
    assert len(previews) == tracking['decoded_frames'], 'Decoded frame count changed; check video/tracking.'
    if tracking['timestamps'] == 'video_POS_MSEC':
        assert all(math.isfinite(t) for t in timestamps)
        assert all(b>a for a,b in zip(timestamps,timestamps[1:]))
        first = timestamps[0]
        timestamps = [t-first for t in timestamps]
    else:
        timestamps = [i/tracking['fps'] for i in range(len(previews))]
    _annotation_cache = dict(key=cache_key, previews=previews, timestamps=timestamps)
    print(f'{len(previews)} previews ready ({sum(map(len,previews))/1e6:.1f} MB).')
else:
    print('Using cached previews; no video decoding needed.')
previews = _annotation_cache['previews']
timestamps = _annotation_cache['timestamps']

frame_slider = widgets.IntSlider(min=0, max=len(previews)-1, value=0, description='Frame', continuous_update=False,
                                layout=widgets.Layout(width='650px'))
event_choice = widgets.Dropdown(options=list(annotations['events']), description='Event')
assign_button = widgets.Button(description='Save + next event', button_style='success')
image_widget = widgets.Image(format='jpeg')
frame_label = widgets.HTML()
summary = widgets.HTML()
selected = {}

def refresh_summary():
    summary.value = '<pre>' + json.dumps(annotations['events'], indent=2) + '</pre>'

def show_frame(change=None):
    index = frame_slider.value
    selected.update(frame=int(index), time_s=float(timestamps[index]))
    image_widget.value = previews[index]
    frame_label.value = f'Frame <b>{index}</b> — {timestamps[index]:.3f} s'

def jump(delta):
    frame_slider.value = max(0,min(len(previews)-1,frame_slider.value+delta))

buttons = []
for delta in [-30,-5,-1,1,5,30]:
    button = widgets.Button(description=f'{delta:+d} frames', layout=widgets.Layout(width='100px'))
    button.on_click(lambda _, delta=delta: jump(delta))
    buttons.append(button)

def select_event(change=None):
    saved = annotations['events'][event_choice.value].get('frame')
    if isinstance(saved,int) and 0 <= saved < len(previews):
        frame_slider.value = saved

def save_event(button):
    annotations['events'][event_choice.value] = dict(selected)
    temporary = events_path.with_suffix('.json.tmp')
    temporary.write_text(json.dumps(annotations,indent=2)+'\n')
    temporary.replace(events_path)
    refresh_summary()
    names = list(annotations['events'])
    index = names.index(event_choice.value)
    if index+1 < len(names): event_choice.value = names[index+1]

frame_slider.observe(show_frame, names='value')
event_choice.observe(select_event, names='value')
assign_button.on_click(save_event)
controls = widgets.HBox(buttons)
_annotation_widgets = [frame_slider,event_choice,assign_button,image_widget,frame_label,summary,controls,*buttons]
display(event_choice,frame_slider,controls,frame_label,image_widget,assign_button,summary)
select_event(); show_frame(); refresh_summary()


In [ ]:
# Run after checking all six events for the selected demo.
batch('approve-events', '--demo', demo)

## Test retargeting with local recording
Record demo_001 under /content/humanoid_retarget_runs, then copy finished outputs and log to Drive. While running, inspect the local log in that folder. Local files are lost if the runtime is reset before copying finishes. Once this episode is validated, remove --demo and demo_001 from the command to process all ready episodes.


In [ ]:
# Check the subprocess environment used by the runner, not just notebook imports.
probe = subprocess.run(
    [sys.executable, '-c', "import importlib.util; names=['libero','robosuite','mujoco','numpy','pandas','scipy','h5py','imageio']; missing=[n for n in names if importlib.util.find_spec(n) is None]; print('Missing: '+', '.join(missing) if missing else 'Dependencies found'); raise SystemExit(bool(missing))"],
    capture_output=True, text=True,
)
print(probe.stdout)
if probe.returncode:
    print(probe.stderr)
    raise RuntimeError('Restore the dependency installation from notebook 05 in this runtime, then rerun this cell. Annotations are already saved on Drive.')
# --retry allows episodes that failed before dependencies were installed to run again.
# Completed unchanged episodes are still skipped.
batch('retarget', '--demo', 'demo_001', '--retry',
      '--work-root', '/content/humanoid_retarget_runs')
batch('status')


## Review one replay
Choose the demo, watch its whole replay, and inspect the report. Approval is separate from simulator success.

In [ ]:
from IPython.display import Video
demo = 'demo_001'
manifest = json.loads((project/'configs/retarget_batch.json').read_text())
entry = manifest['demos'][demo]
print('Batch status:', entry['status'])
if entry.get('error'):
    print(entry['error'])
if not entry.get('run'):
    print('No rollout attempted yet. Approve event annotations and run retargeting first.')
else:
    run = project/entry['run']
    # During execution, files live locally; completed attempts are copied to Drive.
    local_run = Path(entry.get('local_run', '/nonexistent'))
    if not (run/'report.json').exists() and local_run.is_dir():
        run = local_run
    report_file = run/'report.json'
    replay_file = run/'replay.mp4'
    if report_file.exists():
        print(report_file.read_text())
    else:
        print('No report: the process may have failed before recording started, or may still be starting.')
    log = project/'outputs/batch_logs'/f'{demo}_{run.name}.log'
    if not log.exists():
        log = Path(entry.get('local_log', '/nonexistent'))
    if log.exists() and (not report_file.exists() or entry['status']=='retarget_failed'):
        print('Run log:', log)
        print(log.read_text()[-12000:])
    if replay_file.exists() and replay_file.stat().st_size > 0:
        display(Video(str(replay_file), embed=True))
    else:
        print('No replay was produced. Resolve the log error, then retry retargeting.')


In [ ]:
# Set these only AFTER watching the replay. This cell intentionally refuses an unset decision.
decision = None  # 'approve' or 'reject'
note = ''       # Describe the observed grasp, transfer, placement and release.
assert decision in ('approve','reject') and note.strip()
batch('review','--demo',demo,'--decision',decision,'--note',note)


In [ ]:
batch('export-list')
batch('status')


The approved list is ready for a later multi-episode LeRobot export. This notebook does not train or modify the VLA.

## Check path variation and timing
Run after recordings have finished copying to Drive. This reads paths, phases and small state/action arrays only. It does not rerun simulations or change approvals. Compare successful episodes first; failed runs can have incomplete timings. Shape RMS is measured after removing each path’s straight endpoint connection. Simulated time and wall-clock processing time are different.


In [ ]:
import runpy
analysis = runpy.run_path(str(project/'scripts/analyze_retarget_batch.py'))
episode_summary, phase_timing, path_distances = analysis['analyze'](project)
display(episode_summary)
display(phase_timing)
display(path_distances)
